# 00 — Environment check

Confirms Python, MLX (Apple GPU), and that the R2 models are cached locally.
Nothing is downloaded here.

In [1]:
# --- setup: make the repo importable and quiet logs ---
import os, sys, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_OFFLINE", "1")  # models are already cached → fully local
logging.basicConfig(level=logging.WARNING)
print("repo:", ROOT)

repo: /Users/siavash/Projects/articles/narrow-ai-with-kv


In [2]:
import platform, importlib.metadata as md_
import mlx.core as mx
print("python ", platform.python_version(), "|", platform.machine(), platform.mac_ver()[0])
print("mlx    ", md_.version("mlx"), "| device:", mx.default_device())
for pkg in ["mlx-lm", "mlx-whisper", "pydantic", "ipywidgets"]:
    try:
        print(f"{pkg:8}", md_.version(pkg))
    except md_.PackageNotFoundError:
        print(f"{pkg:8} NOT INSTALLED")

python  3.12.15 | arm64 26.5.2
mlx     0.32.4 | device: Device(gpu, 0)
mlx-lm   0.32.2
mlx-whisper 0.4.3
pydantic 2.14.1
ipywidgets 8.1.9


In [3]:
from narrow_ai.settings import load_settings
from narrow_ai.models.hub import local_or_repo
s = load_settings()
for name in [s.model.model_id, s.speech.whisper_model]:
    path = local_or_repo(name)
    print(("cached  " if path != name else "MISSING ") + name)
print("backend:", s.model.backend, "| KV cache mode:", s.model.cache_mode,
      "| constrained:", s.model.constrained)

cached  mlx-community/Qwen3-1.7B-4bit
cached  mlx-community/whisper-small-mlx
backend: mlx | KV cache mode: persistent | constrained: True


In [4]:
import subprocess
mem = int(subprocess.check_output(["sysctl", "-n", "hw.memsize"])) / 2**30
chip = subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"]).decode().strip()
print(f"{chip}, {mem:.0f} GB unified memory")

Apple M4, 16 GB unified memory
